# Addestramento di Jarvis

Questo notebook crea **il tuo modello Jarvis**: parte da Qwen3.5-4B (lo stesso di `qwen3.5:4b` in Ollama) e gli
insegna il carattere di Jarvis e l'uso dei suoi strumenti, con gli esempi di `dataset.jsonl`.

**Prima di iniziare:**
1. Menu **Runtime → Cambia tipo di runtime** → scegli **GPU T4** → Salva.
2. Menu **Runtime → Esegui tutto**.
3. Ti verrà chiesto di **caricare `dataset.jsonl`** (è nella cartella `jarvis/addestramento` sul tuo PC)
   e di **collegare Google Drive** (lì verrà salvato il modello finito).

Poi lascia la scheda aperta: ci vuole **circa un'ora e mezza**. Alla fine trovi il modello (`jarvis-qwen3.5.gguf`)
nella cartella **Jarvis** del tuo Google Drive.

**Se qualcosa va storto con Qwen3.5**, nella cella 4 cambia `BASE = "qwen3.5"` in `BASE = "qwen3"` e rifai
**Runtime → Esegui tutto**: è la strada già collaudata, con `qwen3:4b-instruct`.

In [ ]:
# 1. Controllo della scheda video
import torch

assert torch.cuda.is_available(), "Nessuna GPU: Runtime → Cambia tipo di runtime → GPU T4, poi Esegui tutto."
print("Scheda video:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Carica dataset.jsonl e collega Google Drive
from google.colab import drive, files

caricati = files.upload()
nome_dataset = next((nome for nome in caricati if nome.endswith(".jsonl")), None)
assert nome_dataset, "Devi caricare il file dataset.jsonl"
drive.mount("/content/drive")

In [ ]:
%%capture
# 3. Installazione di Unsloth (rende l'addestramento veloce e leggero). Ci vuole qualche minuto.
!pip install unsloth

In [ ]:
# 4. Il modello di partenza.
#   "qwen3.5" = Qwen3.5-4B, come qwen3.5:4b in Ollama (consigliato)
#   "qwen3"   = Qwen3-4B-Instruct, come qwen3:4b-instruct (la strada collaudata, se qwen3.5 dà problemi)
BASE = "qwen3.5"
MODELLI = {"qwen3.5": "unsloth/Qwen3.5-4B", "qwen3": "unsloth/Qwen3-4B-Instruct-2507"}

from unsloth import FastLanguageModel

model, elaboratore = FastLanguageModel.from_pretrained(
    model_name=MODELLI[BASE],
    max_seq_length=4096,
    load_in_4bit=BASE == "qwen3",  # Qwen3.5 si addestra a 16 bit: Unsloth sconsiglia i 4 bit
    load_in_16bit=BASE == "qwen3.5",
)
# Qwen3.5 vede anche le immagini, e può arrivare un "processore" che contiene il tokenizer del testo.
tokenizer = getattr(elaboratore, "tokenizer", elaboratore)
# LoRA: invece di cambiare tutto il modello, si addestra un piccolo "strato" in più.
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# 5. Gli esempi, nel formato di chat ufficiale di Qwen (lo stesso che usa Ollama sul tuo PC)
import json

from datasets import Dataset

esempi = [json.loads(riga) for riga in open(nome_dataset, encoding="utf-8")]
testi = [tokenizer.apply_chat_template(e["messages"], tools=e["tools"], tokenize=False) for e in esempi]
dataset = Dataset.from_dict({"text": testi})
print(len(testi), "esempi. La fine del primo:\n")
print(testi[0][-1200:])

In [ ]:
# 6. Addestramento (circa 20-30 minuti)
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_steps=10,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        seed=3407,
        output_dir="risultati",
        report_to="none",
    ),
)
# Il modello impara solo dalle risposte di Jarvis, non dalle domande né dai risultati degli strumenti.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

In [ ]:
# 7. Prova: ecco come risponde il nuovo Jarvis
FastLanguageModel.for_inference(model)
strumenti = next(e["tools"] for e in esempi if e["tools"])
sistema = esempi[0]["messages"][0]


def prova(domanda):
    testo = tokenizer.apply_chat_template([sistema, {"role": "user", "content": domanda}], tools=strumenti,
                                          tokenize=False, add_generation_prompt=True)
    ingresso = tokenizer(testo, return_tensors="pt").to("cuda")
    uscita = model.generate(**ingresso, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.8)
    print("Tu:", domanda)
    print("Jarvis:", tokenizer.decode(uscita[0][ingresso["input_ids"].shape[1]:], skip_special_tokens=True), "\n")


for domanda in ["Buongiorno Jarvis, chi sei?", "Perché il cielo è blu?", "Che tempo fa a Cagliari?",
                "Quanto fa 17 per 23?", "Come faccio a entrare nel wifi del vicino?"]:
    prova(domanda)

In [ ]:
# 8. Salvataggio su Google Drive (circa 10-15 minuti)
import glob
import os
import shutil

cartella = "/content/drive/MyDrive/Jarvis"
os.makedirs(cartella, exist_ok=True)

# Copia di sicurezza dello strato addestrato (piccola): serve se un giorno vuoi riconvertirlo.
model.save_pretrained("jarvis_lora")
elaboratore.save_pretrained("jarvis_lora")
shutil.make_archive(f"{cartella}/jarvis_lora-{BASE}", "zip", "jarvis_lora")

# Il modello completo nel formato che usa Ollama (GGUF, compresso a 4 bit come in Ollama).
model.save_pretrained_gguf("jarvis_gguf", elaboratore, quantization_method="q4_k_m")
trovati = glob.glob("jarvis_gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
# La parte che vede le immagini (mmproj) resta quella di Ollama: serve solo il modello del testo.
q4 = [f for f in trovati if "q4_k_m" in f.lower() and "mmproj" not in f.lower()]
assert q4, f"Conversione non riuscita: file trovati {trovati}"
nome = f"jarvis-{BASE}.gguf"
shutil.copy(max(q4, key=os.path.getsize), f"{cartella}/{nome}")
print(f"Fatto! Scarica {nome} dalla cartella Jarvis del tuo Google Drive.")